In [ ]:
import shapely

from geo_data import data_handler, map_style
from geo_data.data import path_settings
from geo_data.projections import Equirectangular
from geo_data.svg_handler import MapSVG

RESULT_DIRECTORY = path_settings.results_dir / "german_rivers"
SIMPLIFICATION = 0.015

BBOX = shapely.geometry.box(5.5, 47.2, 15.5, 55.1)  # bounds by Wikipedia (in lon/lat)

CANVAS_KWARGS = {
    "bounds": BBOX.bounds,
    "width": 500,
    "projection": Equirectangular(y_scale=1.5),  # projection used by Wikipedia
}

In [ ]:
# countries
countries = data_handler.prep_dataset(feature="country", source="ne", resolution=10)
countries.geometry = countries.geometry.simplify(SIMPLIFICATION, preserve_topology=True)
german_vicinity = countries.clip(BBOX)
germany = german_vicinity[german_vicinity["admin"] == "Germany"]
german_vicinity = german_vicinity[~(german_vicinity["admin"] == "Germany")]

# states
all_states = data_handler.prep_dataset(feature="state", source="ne", resolution=10)
states = all_states[all_states["admin"] == "Germany"].copy()
states.geometry = states.geometry.simplify(SIMPLIFICATION, preserve_topology=True)

The river data is preprocessed. See `data_creation/rivers_data_creation.ipynb` for a detailed explanation.

In [ ]:
# get river datasets
rivers = data_handler.prep_dataset(
    feature="river_germany", source="osm", resolution=10, identifier="name"
)
rivers.geometry = rivers.geometry.simplify(SIMPLIFICATION, preserve_topology=True)
# clip to bounding box of germany + vicinity
rivers = rivers.clip(BBOX)

# Germany River Map
This creates a map with all rivers that are saved in the river data (which are the 30 longest rivers that have a part in Germany). This map only draws the rivers and no rivers are highlighted.

In [ ]:
canvas = MapSVG(**CANVAS_KWARGS)
canvas.add_background(map_style.COLORS["lake"])

# styles
state_style = dict(map_style.STYLES["land"])
state_style["stroke_width"] = "0.33"

normal_river_style = dict(map_style.STYLES["river"])
normal_river_style["stroke_width"] = "2"

other_river_style = normal_river_style.copy()
other_river_style["opacity"] = "0.5"

# countries and states
canvas.add_gdf(germany, "Germany", **map_style.STYLES["land"])
canvas.add_gdf(states, "States", **state_style)
canvas.add_gdf(german_vicinity, "Countries", **map_style.STYLES["other"])

# rivers
canvas.add_gdf(rivers, "Rivers", **other_river_style)

# save the original and and optimized version
canvas.save(RESULT_DIRECTORY / "Germany_river_map.svg")
canvas.save(RESULT_DIRECTORY / "optimized" / "Germany_river_map.svg", optimize=True)

canvas

# Mark Rivers
For each river, a canvas the same size as before is drawn, with a single river marked on it. This can be used to lay the svg-file saved here on top of the svg-file of the Germany river map, making it look like the river is marked on the Germany river map.

In [ ]:
normal_river_style = dict(map_style.STYLES["river"])
normal_river_style["stroke_width"] = "2"

marked_river_style = normal_river_style.copy()
marked_river_style["stroke_width"] = "10"
marked_river_style["opacity"] = "0.5"
marked_river_style["stroke"] = map_style.COLORS["highlight"]
marked_river_style["stroke_linecap"] = "round"


for idx, row in rivers.iterrows():
    river_gdf = rivers.loc[[idx]]  # double bracket necessary to get df and not series
    river_name = row["name"]

    canvas = MapSVG(**CANVAS_KWARGS)

    # draw the background-marker
    canvas.add_gdf(river_gdf, "Rivers", **marked_river_style)
    # draw marked river normally
    canvas.add_gdf(river_gdf, "Rivers", **normal_river_style)

    canvas.save(RESULT_DIRECTORY / f"Germany_river_map_{river_name}.svg")
    canvas.save(
        RESULT_DIRECTORY / "optimized" / f"Germany_river_map_{river_name}.svg",
        optimize=True,
    )